In [ ]:
from Libraries.inference_training import Configuration, ImageDataset
from Libraries.inference_training import initCudaEnvironment, createTransforms
from Libraries.inference_training import drawImageAndFeatureMasks
from Libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from Libraries.inference_training import trainModel, saveModel, loadModel
from Libraries.inference_training import createModelInstance, testInference
import os
from paths import TRAIN, TEST

In [ ]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

# base model

In [ ]:
# train on the GPU or on the CPU, if a GPU is not available
config = Configuration()
print("Device: " + str(config.device))

def create_model(trainDirectory, testDirectory):
    """
    Create, train, and export an instance segmentation model using a fixed configuration setup.

    This function initializes a model configuration with hardcoded values, loads training and test
    datasets, trains the model, performs evaluation on a sample test image, and exports the model
    to both PyTorch and ONNX formats.

    Parameters:
    -----------
    trainDirectory : str
        Path to the training dataset directory containing images and annotation files.
    testDirectory : str
        Path to the testing dataset directory used for validation during training.

    Returns:
    --------
    None

    Notes:
    ------
    - The function assumes a fixed model name, label structure, and number of epochs (25).
    - Legend entries, input sizes, ONNX metadata, and training hyperparameters are hardcoded.
    - The resulting model is saved to disk in both PyTorch (.pt) and ONNX (.onnx) formats.
    - A sample inference is performed on image index 88 from the test set.
    - This function uses a global `Configuration` class from the `inference_training` module.
    """
    config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
    config.setFilePrefix("")
    config.setModelName("25_epoch_combination_overlay")
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.setEpochs(25)
    description = "Model description"
    config.setOnnxInfo(producer="Tygron", description=description)
    
    config.addLegendEntry("Background", 0, "#00000000")
    config.addLegendEntry("Label name 1", 1, "#00ffbf")
    config.addLegendEntry("Label name 2", 2, "#12d900")
    
    config.setOnnxMetaData(scoreThreshold=0.2,
                           maskThreshold=0.3,
                           strideFraction=0.5)
    
    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    trainingDataset = ImageDataset(config, True, createTransforms(False))
    testDataset = ImageDataset(config, False, createTransforms(False))
    
    print("Train Image count: "+str(trainingDataset.__len__()))
    print("Test Image count: "+str(testDataset.__len__()))
    
    if not trainingDataset.validateFiles(False):
        print("Inconsistent training dataset ")
        trainingDataset.validateFiles(True)
    
    if not testDataset.validateFiles(False):
        print("Inconsistent test dataset ")
        testDataset.validateFiles(True)
    
    print("Pytorch model name " + config.getPytorchModelFileName())
    print("Onnx file name " + config.getOnnxFileName())
    
    imageNumber = 5
    print(trainingDataset.getLabelList(imageNumber))
    drawImageAndFeatureMasks(config, trainingDataset, imageNumber)
    
    model = trainModel(config, trainingDataset, testDataset)
    saveModel(config, model, path="models/"+config.getPytorchModelFileName())
    
    model.eval()
    testPrediction = testInference(config, model=model,
                               dataset=testDataset, imageNumber=88)
    
    exportOnnxModel(config, model)
    writeONNXMeta(config)
    onnx_model = loadONNX(config)
    print(f"metadata_props={onnx_model.metadata_props}")

In [ ]:
#create_model(TRAIN, TEST)

In [ ]:
#loadExistingModel = False

#if loadExistingModel:
#    model = createModelInstance(config)
#    loadModel(config, model, path=config.getPytorchModelFileName())

#else:
#    model = trainModel(config, trainingDataset, testDataset)
#    saveModel(config, model, path=config.getPytorchModelFileName())

In [ ]:
model = createModelInstance(config)
config.setDatasetPaths(trainPath=TRAIN, testPath=TEST)
testDataset = ImageDataset(config, False, createTransforms(False))
loadModel(config, model, path=config.getPytorchModelFileName())
model.eval()
testPrediction = testInference(config, model=model,
                               dataset=testDataset, imageNumber=20)

In [ ]:
#exportOnnxModel(config, model)
#writeONNXMeta(config)
#onnx_model = loadONNX(config)
#print(f"metadata_props={onnx_model.metadata_props}")